# conv-kernel-shape — worked example 2: Construct an OC-first Conv2d weight from loose dims

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-kernel-shape`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

An `nn.Conv2d` weight tensor has shape **`(OC, IC, KH, KW)`** — output channels first. The constructor signature is `(in_channels, out_channels, kernel_size)`, so the argument order and the tensor axis order disagree on where IC vs OC sit. Building the tensor yourself is the cleanest way to drill that the saved weight is OC-first.

## Worked solution

**Step 1 — pull the four scalars.** From the dims tuple `(ic, oc, kh, kw) = (3, 8, 3, 5)` we read IC=3, OC=8, KH=3, KW=5. Note the input tuple deliberately lists IC before OC (constructor order) to test that we re-order correctly.

**Step 2 — choose the tensor axis order.** The weight must be `(OC, IC, KH, KW) = (8, 3, 3, 5)`. If we instead wrote `(IC, OC, KH, KW)` we'd produce the **ConvTranspose2d** layout and the shape check against a real `nn.Conv2d` would fail.

**Step 3 — fill, not random.** `t.arange` over `OC*IC*KH*KW` reshaped to the target shape gives a deterministic, inspectable tensor of the right size. We cast to `float32` because conv weights are fp32 by default.

**Step 4 — verify against the real module.** We instantiate `nn.Conv2d(in_channels=3, out_channels=8, kernel_size=(3,5))` and confirm `weight.shape` matches exactly. The numel is `8*3*3*5 = 360`.

In [ ]:
def build_conv2d_weight(dims) -> Tensor:
    IC, OC, KH, KW = dims
    n = OC * IC * KH * KW
    return t.arange(n, dtype=t.float32).reshape(OC, IC, KH, KW)

t.manual_seed(0)
dims = (3, 8, 3, 5)  # (IC, OC, KH, KW) -- constructor order in, OC-first tensor out
w = build_conv2d_weight(dims)
ref = t.nn.Conv2d(in_channels=3, out_channels=8, kernel_size=(3, 5))
print(tuple(w.shape), tuple(w.shape) == tuple(ref.weight.shape), w.numel())